In [0]:
# Databricks notebook source
# =============================================================
#  GEO SAFRAS · 02_gs_silver_transform.py
#  Transformação Bronze → Silver
#  • Normalização e tipagem
#  • Deduplicação com MERGE
#  • Colunas de auditoria (created_at / updated_at)
#  • Tratamento de dados inválidos documentado
# =============================================================

# COMMAND ----------
from pyspark.sql import functions as F
from pyspark.sql.types import *
from delta.tables import DeltaTable
from datetime import datetime

CATALOG  = "workspace"
SCHEMA_B = "gs_bronze"
SCHEMA_S = "gs_silver"
NOW      = F.lit(datetime.now().isoformat()).cast("timestamp")

print(f"[Silver] Início: {datetime.now().isoformat()}")

# COMMAND ----------
# ═══════════════════════════════════════════════════════════════
#  1. PEDIDOS Bronze → Silver
# ═══════════════════════════════════════════════════════════════

# COMMAND ----------
df_b = spark.table(f"{CATALOG}.{SCHEMA_B}.pedidos")

df_s_pedidos = (df_b
    # Normalização textual
    .withColumn("cultura",    F.initcap(F.trim(F.col("cultura"))))
    .withColumn("cidade",     F.lower(F.trim(F.col("cidade"))))
    .withColumn("localidade", F.lower(F.trim(F.col("localidade"))))
    .withColumn("estado",     F.upper(F.trim(F.col("estado"))))
    .withColumn("safra",      F.trim(F.col("safra")))
    .withColumn("subcultura", F.lower(F.trim(F.col("subcultura"))))
    .withColumn("subcultura2",F.lower(F.trim(F.col("subcultura2"))))

    # Tipagem numérica segura
    .withColumn("vendas",              F.col("vendas").cast("decimal(14,2)"))
    .withColumn("quantidade",          F.col("quantidade").cast("decimal(10,2)"))
    .withColumn("desconto",            F.col("desconto").cast("decimal(5,4)"))
    .withColumn("lucro",               F.col("lucro").cast("decimal(14,2)"))
    .withColumn("custo_total",         F.col("custo_total").cast("decimal(14,2)"))
    .withColumn("margem_liquida_pct",  F.col("margem_liquida_pct").cast("decimal(6,2)"))
    .withColumn("preco_por_unidade",   F.col("preco_por_unidade").cast("decimal(10,2)"))
    .withColumn("custo_por_unidade",   F.col("custo_por_unidade").cast("decimal(10,2)"))
    .withColumn("lucro_por_unidade",   F.col("lucro_por_unidade").cast("decimal(10,2)"))
    .withColumn("impacto_desconto",    F.col("impacto_desconto").cast("decimal(14,2)"))
    .withColumn("custo_insumos_est",        F.col("custo_insumos_est").cast("decimal(14,2)"))
    .withColumn("custo_fertilizantes_est",  F.col("custo_fertilizantes_est").cast("decimal(14,2)"))
    .withColumn("custo_defensivos_est",     F.col("custo_defensivos_est").cast("decimal(14,2)"))
    .withColumn("custo_outros_est",         F.col("custo_outros_est").cast("decimal(14,2)"))
    .withColumn("cepea_referencia_sc",  F.col("cepea_referencia_sc").cast("decimal(10,2)"))
    .withColumn("gap_preco_cepea",      F.col("gap_preco_cepea").cast("decimal(10,2)"))
    .withColumn("perc_vs_cepea",        F.col("perc_vs_cepea").cast("decimal(8,4)"))
    .withColumn("receita_perdida_cepea",F.col("receita_perdida_cepea").cast("decimal(14,2)"))
    .withColumn("lucro_justo_cepea",    F.col("lucro_justo_cepea").cast("decimal(14,2)"))

    # Auditoria
    # NOTA: preco_realizado_sc e margem_bruta_pct são GENERATED ALWAYS AS
    # na tabela Silver — o Delta Lake as calcula automaticamente.
    # Não inserir manualmente para evitar DELTA_VIOLATE_CONSTRAINT.
    .withColumn("updated_at", NOW)
    .withColumn("source_file", F.col("source_file"))

    # Remover colunas de ingestão Bronze não necessárias no Silver
    .drop("ingestion_timestamp")
)

# Validação: registros sem id_pedido
invalidos = df_s_pedidos.filter(F.col("id_pedido").isNull()).count()
if invalidos > 0:
    print(f"  ⚠️  {invalidos} registros sem id_pedido ignorados no Silver")
df_s_pedidos = df_s_pedidos.filter(F.col("id_pedido").isNotNull())

total = df_s_pedidos.count()
print(f"  ✓ pedidos Silver: {total} registros prontos")

# MERGE Silver — chave: id_pedido
# Colunas que vêm do source — excluir GENERATED ALWAYS AS
# (preco_realizado_sc, margem_bruta_pct)
cols_pedidos = {c: f"src.{c}" for c in df_s_pedidos.columns}

if spark.catalog.tableExists(f"{CATALOG}.{SCHEMA_S}.pedidos"):
    dt = DeltaTable.forName(spark, f"{CATALOG}.{SCHEMA_S}.pedidos")
    (dt.alias("tgt")
       .merge(df_s_pedidos.alias("src"), "tgt.id_pedido = src.id_pedido")
       .whenMatchedUpdate(set=cols_pedidos)
       .whenNotMatchedInsert(values=cols_pedidos)
       .execute())
    print("  ✓ pedidos Silver: MERGE concluído")
else:
    df_s_pedidos.write.format("delta").mode("overwrite").saveAsTable(f"{CATALOG}.{SCHEMA_S}.pedidos")
    print("  ✓ pedidos Silver: criada e carregada")

# Atualizar colunas derivadas de pedidos
spark.sql(f"UPDATE {CATALOG}.{SCHEMA_S}.pedidos SET preco_realizado_sc = CASE WHEN quantidade > 0 THEN ROUND(vendas / (quantidade * 1000 / 60), 2) ELSE NULL END, margem_bruta_pct = CASE WHEN vendas > 0 THEN ROUND((lucro / vendas) * 100, 2) ELSE NULL END")
print("  ✓ pedidos Silver: derivadas atualizadas")

# COMMAND ----------
# ═══════════════════════════════════════════════════════════════
#  2. CALENDARIO_SAFRAS Bronze → Silver
# ═══════════════════════════════════════════════════════════════

# COMMAND ----------
df_b_cal = spark.table(f"{CATALOG}.{SCHEMA_B}.calendario_safras")

df_s_cal = (df_b_cal
    .withColumn("cultura",    F.initcap(F.trim(F.col("cultura"))))
    .withColumn("talhao",     F.trim(F.col("talhao")))
    .withColumn("safra_tipo", F.lower(F.trim(F.col("safra_tipo"))))
    .withColumn("mes_nome",   F.trim(F.col("mes_nome")))
    .withColumn("estagio",    F.initcap(F.trim(F.col("estagio"))))
    .withColumn("cor_hex",    F.upper(F.trim(F.col("cor_hex"))))

    # Datas
    .withColumn("mes_data_inicio",    F.col("mes_data_inicio").cast("date"))
    .withColumn("mes_data_fim",       F.col("mes_data_fim").cast("date"))
    .withColumn("data_inicio_estagio",F.col("data_inicio_estagio").cast("date"))
    .withColumn("data_fim_estagio",   F.col("data_fim_estagio").cast("date"))

    # Auditoria
    # NOTA: duracao_estagio_dias é GENERATED ALWAYS AS — calculada automaticamente.
    .withColumn("updated_at", NOW)
    .drop("ingestion_timestamp")
)

print(f"  ✓ calendario_safras Silver: {df_s_cal.count()} registros prontos")

chave_cal = """
    tgt.cultura    = src.cultura    AND
    tgt.talhao     = src.talhao     AND
    tgt.ano        = src.ano        AND
    tgt.safra_tipo = src.safra_tipo AND
    tgt.mes_num    = src.mes_num
"""

# Colunas que vêm do source — excluir GENERATED ALWAYS AS
# (duracao_estagio_dias)
cols_cal = {c: f"src.{c}" for c in df_s_cal.columns}

if spark.catalog.tableExists(f"{CATALOG}.{SCHEMA_S}.calendario_safras"):
    dt = DeltaTable.forName(spark, f"{CATALOG}.{SCHEMA_S}.calendario_safras")
    (dt.alias("tgt")
       .merge(df_s_cal.alias("src"), chave_cal)
       .whenMatchedUpdate(set=cols_cal)
       .whenNotMatchedInsert(values=cols_cal)
       .execute())
    print("  ✓ calendario_safras Silver: MERGE concluído")
else:
    df_s_cal.write.format("delta").mode("overwrite").saveAsTable(f"{CATALOG}.{SCHEMA_S}.calendario_safras")
    print("  ✓ calendario_safras Silver: criada e carregada")

# Atualizar coluna derivada de calendario
spark.sql(f"UPDATE {CATALOG}.{SCHEMA_S}.calendario_safras SET duracao_estagio_dias = CASE WHEN data_inicio_estagio IS NOT NULL AND data_fim_estagio IS NOT NULL THEN DATEDIFF(data_fim_estagio, data_inicio_estagio) ELSE NULL END")
print("  ✓ calendario_safras Silver: derivadas atualizadas")

# COMMAND ----------
# ═══════════════════════════════════════════════════════════════
#  3. SAUDE_FINANCEIRA Bronze → Silver
# ═══════════════════════════════════════════════════════════════

# COMMAND ----------
df_b_sf = spark.table(f"{CATALOG}.{SCHEMA_B}.saude_financeira")

df_s_sf = (df_b_sf
    .withColumn("fazenda", F.lower(F.trim(F.col("fazenda"))))

    # Tipagem numérica
    .withColumn("receita_total",  F.col("receita_total").cast("decimal(16,2)"))
    .withColumn("custo_total",    F.col("custo_total").cast("decimal(16,2)"))
    .withColumn("lucro_total",    F.col("lucro_total").cast("decimal(16,2)"))
    .withColumn("passivo_circulante_base",   F.col("passivo_circulante_base").cast("decimal(16,2)"))
    .withColumn("ativo_circulante_base",     F.col("ativo_circulante_base").cast("decimal(16,2)"))
    .withColumn("capital_giro_base",         F.col("capital_giro_base").cast("decimal(16,2)"))
    .withColumn("liquidez_corrente_base",    F.col("liquidez_corrente_base").cast("decimal(6,2)"))
    .withColumn("ativo_total_simulado",      F.col("ativo_total_simulado").cast("decimal(18,2)"))
    .withColumn("passivo_nao_circulante_simulado", F.col("passivo_nao_circulante_simulado").cast("decimal(16,2)"))
    .withColumn("endividamento_geral_pct",   F.col("endividamento_geral_pct").cast("decimal(6,2)"))
    .withColumn("despesas_financeiras_simuladas", F.col("despesas_financeiras_simuladas").cast("decimal(16,2)"))
    .withColumn("cobertura_juros_x",         F.col("cobertura_juros_x").cast("decimal(6,2)"))
    .withColumn("score_risco_financeiro",    F.col("score_risco_financeiro").cast("int"))

    # Auditoria
    # NOTA: margem_liquida_pct, receita_por_ha, lucro_por_ha, custo_por_ha
    # são GENERATED ALWAYS AS na tabela Silver — calculadas automaticamente.
    .withColumn("updated_at", NOW)
    .drop("ingestion_timestamp")
)

print(f"  ✓ saude_financeira Silver: {df_s_sf.count()} registros prontos")

chave_sf = "tgt.fazenda = src.fazenda AND tgt.ano = src.ano"

# Colunas que vêm do source — excluir GENERATED ALWAYS AS
# (margem_liquida_pct, receita_por_ha, lucro_por_ha, custo_por_ha)
cols_sf = {c: f"src.{c}" for c in df_s_sf.columns}

if spark.catalog.tableExists(f"{CATALOG}.{SCHEMA_S}.saude_financeira"):
    dt = DeltaTable.forName(spark, f"{CATALOG}.{SCHEMA_S}.saude_financeira")
    (dt.alias("tgt")
       .merge(df_s_sf.alias("src"), chave_sf)
       .whenMatchedUpdate(set=cols_sf)
       .whenNotMatchedInsert(values=cols_sf)
       .execute())
    print("  ✓ saude_financeira Silver: MERGE concluído")
else:
    df_s_sf.write.format("delta").mode("overwrite").saveAsTable(f"{CATALOG}.{SCHEMA_S}.saude_financeira")
    print("  ✓ saude_financeira Silver: criada e carregada")

# Atualizar colunas derivadas de saude_financeira
spark.sql(f"UPDATE {CATALOG}.{SCHEMA_S}.saude_financeira SET margem_liquida_pct = CASE WHEN receita_total > 0 THEN ROUND((lucro_total / receita_total) * 100, 2) ELSE NULL END, receita_por_ha = CASE WHEN area_ha > 0 THEN ROUND(receita_total / area_ha, 2) ELSE NULL END, lucro_por_ha = CASE WHEN area_ha > 0 THEN ROUND(lucro_total / area_ha, 2) ELSE NULL END, custo_por_ha = CASE WHEN area_ha > 0 THEN ROUND(custo_total / area_ha, 2) ELSE NULL END")
print("  ✓ saude_financeira Silver: derivadas atualizadas")

# COMMAND ----------
print(f"\n[Silver] ✅ Transformação concluída em {datetime.now().isoformat()}")
print(f"  Tabelas atualizadas:")
print(f"  • {CATALOG}.{SCHEMA_S}.pedidos")
print(f"  • {CATALOG}.{SCHEMA_S}.calendario_safras")
print(f"  • {CATALOG}.{SCHEMA_S}.saude_financeira")